In [0]:
%run "/Workspace/Users/jeevan.azureacc3@gmail.com/bankaml-de-project/notebooks/04_utils/watermark_incremental_load"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
branches_bronze_df = spark.read.table("bankaml.bronze.branches")
branches_bronze_df = get_new_rows_from_source_df(branches_bronze_df, "silver", "silver", "branches")

branches_quarantine_df = branches_bronze_df.filter(
    col("branch_id").isNull() | col("branch_name").isNull()
)

branches_bronze_df = branches_bronze_df.filter(col("branch_id").isNotNull() & col("branch_name").isNotNull()).dropDuplicates(["branch_id"])

## if region_code == null then map country → region_code using the same REGIONS. So finding the preferred_region for each country
region_code_count_df = branches_bronze_df.filter(col("region_code").isNotNull()) \
    .groupBy(col("country"), col("region_code")) \
    .agg(count("branch_id").alias("region_count"))

window_logic = Window.partitionBy(col("country")).orderBy(
    col("region_count").desc(), col("region_code").asc()
)
preferred_region_df = region_code_count_df.withColumn(
    "priority_region", row_number().over(window_logic)
).filter(col("priority_region") == 1)

preferred_region_df = preferred_region_df.select(
    "country", col("region_code").alias("preferred_region")
)

## While ingesting the data into bronze delta table all columns are read as StringType. So now need to cast columns
branches_bronze_df = branches_bronze_df.withColumn(
    "created_at", col("created_at").cast(TimestampType())
)
## Mapping preferred_region based on country if region_code == null
branches_bronze_df = (
    branches_bronze_df.join(preferred_region_df, "country", "left")
    .withColumn("region_code", coalesce("region_code", "preferred_region"))
    .drop("preferred_region")
)

branches_silver_df = branches_bronze_df.withColumns(
    {"country": upper(col("country")), "region_code": upper(col("region_code"))}
).select(
    "branch_id",
    "branch_name",
    "city",
    "country",
    "region_code",
    "created_at",
    "created_by",
    "_ingestion_ts",
)

In [0]:
%sql
create table if not exists bankaml.silver.branches
(
    branch_id string,
    branch_name string,
    city string,
    country string,
    region_code string,
    created_at timestamp,
    created_by string,
    _ingestion_ts timestamp
)
using delta;

In [0]:
target_table = DeltaTable.forName(spark, "bankaml.silver.branches")

(
    target_table.alias("t")
    .merge(
        branches_silver_df.alias("s"), 
        "t.branch_id=s.branch_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
update_last_processed_value(branches_bronze_df, "silver", "silver", "branches")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bankaml.quarantine.branches (
    branch_id STRING,
    branch_name STRING,
    city STRING,
    country STRING,
    region_code STRING,
    created_at STRING,
    created_by STRING,
    _ingestion_ts TIMESTAMP,
    quarantine_reason STRING,  
    quarantined_at TIMESTAMP,
    source_layer STRING
)
using delta;

In [0]:
branches_quarantine_df = branches_quarantine_df.withColumns(
    {
        "quarantine_reason": when(
            col("branch_id").isNull() & col("branch_name").isNull(),
            lit("null branch_id and null branch_name")
        )
        .when(col("branch_id").isNull(), lit("null branch_id"))
        .when(col("branch_name").isNull(), lit("null branch_name"))
        .otherwise(lit("null branch_name")), 
        "quarantined_at": current_timestamp(),
        "source_layer": lit("silver")
    }
)
branches_quarantine_df.write.format("delta").mode("append").saveAsTable("bankaml.quarantine.branches")